# 02 — Feature Engineering

Builds and inspects the full feature set used by the model.
All production logic lives in `src/features.py` — this notebook imports from it and visualises the output.

**5 feature groups → 10,131 total features:**
- TF-IDF word n-grams (1–3): 10,000
- SQL keyword counts: 76
- Special char counts + ratios: 32
- Structural / length: 10
- Entropy: 2
- Obfuscation patterns: 11

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")   # allow src/ imports from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import load_npz

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

train_df = pd.read_csv("../data/processed/train.csv")
val_df   = pd.read_csv("../data/processed/val.csv")
test_df  = pd.read_csv("../data/processed/test.csv")

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    sqli_pct = df["Label"].mean() * 100
    print(f"{name:<6}: {len(df):,} rows  |  {sqli_pct:.1f}% SQLi")

## Group 1 — SQL Keyword Counts

Each keyword occurrence in a query becomes one feature. These are the strongest individual signals.

In [ ]:
from src.features import SQL_KEYWORDS, count_sql_keywords

sample_sqli   = "' UNION SELECT username, password FROM admin--"
sample_benign = "SELECT name FROM products WHERE category = 'Electronics'"

print("SQLi payload keyword counts:")
sqli_counts = {k:v for k,v in count_sql_keywords(sample_sqli).items() if v > 0}
print(sqli_counts)

print("\nBenign query keyword counts:")
benign_counts = {k:v for k,v in count_sql_keywords(sample_benign).items() if v > 0}
print(benign_counts)

## Group 2 — Special Character Ratios

In [ ]:
from src.features import special_char_features

print("SQLi special chars:")
sqli_sc = {k:round(v,4) for k,v in special_char_features(sample_sqli).items() if v > 0}
print(sqli_sc)

print("\nBenign special chars:")
benign_sc = {k:round(v,4) for k,v in special_char_features(sample_benign).items() if v > 0}
print(benign_sc)

## Group 3 — Structural Features

In [ ]:
from src.features import structural_features

print("SQLi structural features:")
print(structural_features(sample_sqli))

print("\nBenign structural features:")
print(structural_features(sample_benign))

## Group 4 — Entropy

In [ ]:
from src.features import entropy_features, shannon_entropy

test_cases = [
    ("SQLi payload",        "' UNION SELECT username, password FROM admin--"),
    ("Benign SQL",          "SELECT name FROM products WHERE category = 'Electronics'"),
    ("Natural language",    "The quick brown fox jumped over the lazy dog"),
    ("Hex obfuscation",     "0x53454c454354202a2046524f4d207573657273"),
    ("Time-based blind",    "1; WAITFOR DELAY '0:0:5'--"),
]

print(f"{'Sample':<25} {'Entropy (overall)':>18} {'Entropy (alnum)':>16}")
print("-" * 63)
for name, query in test_cases:
    e = entropy_features(query)
    print(f"{name:<25} {e['entropy_overall']:>18.4f} {e['entropy_alnum']:>16.4f}")

## Group 5 — Obfuscation Patterns

In [ ]:
from src.features import obfuscation_features, OBFUSCATION_PATTERNS

obfuscated_samples = [
    "SeLeCt * fRoM users",
    "SELECT/*comment*/username FROM users",
    "SELECT CHAR(83,69,76,69,67,84)",
    "SELECT 0x53454c454354",
    "1 OR 1=1",
    "SELECT * FROM users; DROP TABLE users;--",
    "' OR SLEEP(5)--",
    "' UNION ALL SELECT NULL--",
]

print(f"{'Pattern':<22}", end="")
for s in obfuscated_samples:
    print(f"{s[:18]:<20}", end="")
print()
print("-" * (22 + 20*len(obfuscated_samples)))

for pattern_name in OBFUSCATION_PATTERNS:
    print(f"{pattern_name:<22}", end="")
    for s in obfuscated_samples:
        val = obfuscation_features(s)[pattern_name]
        print(f"{'✓' if val else '·':<20}", end="")
    print()

## Load Feature Matrices

> ⚠️ **Data leakage reminder:** The TF-IDF vectorizer was fit on training data only. `val` and `test` matrices were produced with `transform()` only — never `fit_transform()`.

In [ ]:
import json

X_train = load_npz("../data/processed/features_train.npz")
X_val   = load_npz("../data/processed/features_val.npz")
X_test  = load_npz("../data/processed/features_test.npz")
y_train = np.load("../data/processed/labels_train.npy")
y_val   = np.load("../data/processed/labels_val.npy")
y_test  = np.load("../data/processed/labels_test.npy")

with open("../data/processed/feature_names.json") as f:
    feature_names = json.load(f)

tfidf_feats = [n for n in feature_names if n.startswith("tfidf_")]
hc_feats    = [n for n in feature_names if not n.startswith("tfidf_")]

print(f"X_train shape     : {X_train.shape}")
print(f"TF-IDF features   : {len(tfidf_feats):,}")
print(f"Hand-crafted feats: {len(hc_feats):,}")
print(f"Total features    : {len(feature_names):,}")
print(f"Matrix sparsity   : {100*(1 - X_train.nnz/(X_train.shape[0]*X_train.shape[1])):.2f}%")

## Feature Distribution — SQLi vs Benign

In [ ]:
from src.features import queries_to_handcrafted_matrix

hc_train = queries_to_handcrafted_matrix(train_df["Query"])
hc_train["Label"] = y_train

key_features = ["char_len", "entropy_overall", "entropy_alnum",
                "avg_word_len", "digit_ratio", "punct_ratio",
                "sc_count_single_quote", "sc_count_dash_dash",
                "kw_union", "kw_select"]

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for ax, feat in zip(axes.flat, key_features):
    for label, color, name in [(0,"#4A90D9","Benign"),(1,"#E05C5C","SQLi")]:
        vals = hc_train[hc_train["Label"]==label][feat].clip(
            upper=hc_train[feat].quantile(0.98))
        ax.hist(vals, bins=40, alpha=0.55, color=color, label=name, edgecolor="none")
    ax.set_title(feat.replace("_"," "), fontsize=9, fontweight="bold")
    ax.legend(fontsize=7)

plt.suptitle("Feature distributions — SQLi vs Benign (training set)", fontweight="bold", y=1.01)
plt.tight_layout(); plt.show()

## Summary

- **10,131 features** total across 5 groups
- TF-IDF captures raw token patterns; hand-crafted features capture structure and intent
- `entropy_alnum` and `avg_word_len` are the strongest structural separators
- Obfuscation patterns catch evasion attempts that keyword matching misses
- ⚠️ Vectorizer fit on train only — no leakage into val/test

➡️ Next: `03_model_training.ipynb`